# Clase 5 - Cuantización, PEFT/LoRA, QLoRA, Ollama y despliegue de LLMs

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

> Para usarlo en Google Colab: sube este notebook a Colab o guárdalo en Drive y ábrelo desde `Archivo > Abrir notebook`.

**Objetivos.** Al terminar deberías poder:

- Cargar un modelo pequeño de Hugging Face y ejecutar inferencia controlando parámetros de generación.
- Estimar memoria por precisión numérica y comparar parámetros entrenables con LoRA.
- Probar cuantización con `bitsandbytes`.
- Entrenar un adaptador LoRA pequeño.
- Usar Ollama como servidor local mediante API HTTP.
- Construir un mini flujo RAG local con embeddings de Ollama.

**Nota:** Ollama es excelente para inferencia local y prototipos, pero para entrenar LoRA/QLoRA normalmente usarás PyTorch + Transformers + PEFT.

## Plataformas y fuentes de modelos

| Opción | Mejor para | De dónde salen los modelos |
|---|---|---|
| Hugging Face Hub | Entrenar, adaptar, evaluar, descargar checkpoints | https://huggingface.co/models |
| Ollama | Inferencia local rápida por CLI/API | https://ollama.com/library |

Fuentes recomendadas para empezar:

- **Ollama Library:** `llama3.2:1b`, `qwen2.5:0.5b`, `gemma3:1b`, `smollm2:135m`, `all-minilm` para embeddings.
- **Hugging Face:** busca por tarea, licencia, idioma, tamaño y formato.
- **Organizaciones oficiales:** Meta/Llama, Mistral, Google/Gemma, Qwen, Microsoft/Phi, Hugging FaceTB/SmolLM.

No descargues modelos grandes en este laboratorio salvo que sepas cuánta RAM/VRAM tienes disponible.

In [ ]:
#%pip install -q -U transformers accelerate peft bitsandbytes requests pandas openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Si necesitas acceder a modelos privados o con licencia, usa un token.
# from huggingface_hub import login
# login(token="hf_tu_token_aqui")

In [1]:
import gc
import json
import math
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(42)

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Sistema: {platform.platform()}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16 soportado: {torch.cuda.is_bf16_supported()}")
else:
    print("Nota: en CPU se puede ejecutar la ruta pequeña, pero QLoRA/INT8 puede omitirse.")

Python: 3.8.20
PyTorch: 2.4.1+cu121
Sistema: Linux-5.15.0-139-generic-x86_64-with-glibc2.17
CUDA disponible: True
GPU: NVIDIA RTX A6000
BF16 soportado: True


## OpenRouter como proveedor de modelos

OpenRouter permite llamar modelos alojados mediante una API compatible con OpenAI.

Pasos:

1. Crea una cuenta en https://openrouter.ai/.
2. Entra a **Keys** y crea una API key.
4. Elige un modelo desde el [catálogo de OpenRouter](https://openrouter.ai/models$0). El identificador suele tener formato `proveedor/modelo` o un alias como `meta-llama/llama-3.1-8b-instruct`.

OpenRouter también permite consumir modelos ya desplegados y comparar experiencia de API, latencia percibida, costo y calidad de respuesta.


In [ ]:
import getpass
import requests

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Local
# OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "meta-llama/llama-3.1-8b-instruct")

# Opción local/servidor/Colab: define la variable sin mostrarla.
OPENROUTER_API_KEY = ""
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct"


def openrouter_disponible() -> bool:
    return bool(OPENROUTER_API_KEY)


def crear_cliente_openrouter():
    """Crea un cliente con una interfaz de programación similar a la API de OpenAI, pero apuntando a OpenRouter."""
    from openai import OpenAI

    api_key = OPENROUTER_API_KEY
    if not api_key:
        raise RuntimeError(
            "Falta OPENROUTER_API_KEY."
        )

    return OpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=api_key,
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com/",
            "X-OpenRouter-Title": "Curso agentes",
        },
    )


def chat_openrouter(mensajes, modelo=None, temperatura=0.2, max_tokens=350):
    """Envía mensajes estilo OpenAI Chat Completions mediante OpenRouter."""
    cliente = crear_cliente_openrouter()
    respuesta = cliente.chat.completions.create(
        model=modelo or OPENROUTER_MODEL,
        messages=mensajes,
        temperature=temperatura,
        max_tokens=max_tokens,
    )
    return respuesta.choices[0].message.content


def listar_modelos_openrouter(filtro=None, limite=20):
    """Lista modelos disponibles. Si hay API key, la usa; si no, intenta consulta pública."""
    headers = {}
    if openrouter_disponible():
        headers["Authorization"] = f"Bearer {'OPENROUTER_API_KEY'}"
    r = requests.get(f"{OPENROUTER_BASE_URL}/models", headers=headers, timeout=30)
    r.raise_for_status()
    datos = r.json().get("data", [])
    filas = []
    for m in datos:
        mid = m.get("id", "")
        nombre = m.get("name", "")
        if filtro and filtro.lower() not in (mid + " " + nombre).lower():
            continue
        filas.append({
            "id": mid,
            "nombre": nombre,
            "contexto": m.get("context_length"),
            "entrada_usd": m.get("pricing", {}).get("prompt"),
            "salida_usd": m.get("pricing", {}).get("completion"),
        })
        if len(filas) >= limite:
            break
    return pd.DataFrame(filas)

print("OpenRouter configurado. Modelo por defecto:", OPENROUTER_MODEL)


OpenRouter configurado. Modelo por defecto: meta-llama/llama-3.1-8b-instruct


In [3]:
prompt = """
Explica en español, en máximo 6 viñetas, cuándo elegirías:
1. inferencia local cuantizada,
2. LoRA/QLoRA,
3. una API hospedada como OpenRouter.
Incluye una advertencia sobre costos o privacidad.
""".strip()

if openrouter_disponible():
    respuesta = chat_openrouter([
        {"role": "system", "content": "Eres un asistente técnico para una clase de LLMs. Responde claro y directo."},
        {"role": "user", "content": prompt},
    ], temperatura=0.2, max_tokens=400)
    print(respuesta)

# Para explorar modelos disponibles, descomenta:
#listar_modelos_openrouter(filtro="openai", limite=10)


**Elegir la tecnología adecuada**

Aquí te presento las opciones para elegir entre la inferencia local cuantizada, LoRA/QLoRA y una API hospedada como OpenRouter:

*   **Inferencia local cuantizada**: Esta opción es adecuada cuando necesitas procesar datos de manera rápida y segura en tu dispositivo. La inferencia local cuantizada utiliza algoritmos de aprendizaje automático (AIA) en un dispositivo local, lo que reduce la dependencia de la red y la necesidad de conectividad. Sin embargo, puede requerir más recursos y potencia de procesamiento, lo que puede aumentar el costo.

*   **LoRA/QLoRA**: Esta opción es adecuada cuando necesitas reducir la complejidad de los modelos de AIA y mejorar la eficiencia en dispositivos con recursos limitados. LoRA (Large Model in Small Embeddings) y QLoRA (Quantized LoRA) son técnicas que permiten reducir el tamaño de los modelos de AIA y mejorar su eficiencia en dispositivos con recursos limitados. Sin embargo, pueden requerir más recursos y potencia 

In [4]:
listar_modelos_openrouter(filtro="openai", limite=10)

,id,nombre,contexto,entrada_usd,salida_usd
0,openai/gpt-5.6-luna-pro,OpenAI: GPT-5.6 Luna Pro,1050000,0.0000001,0.0000006000000000000001
1,openai/gpt-5.6-luna,OpenAI: GPT-5.6 Luna,1050000,0.0000001,0.0000006000000000000001
2,openai/gpt-5.6-terra-pro,OpenAI: GPT-5.6 Terra Pro,1050000,0.00000100000000000000015,0.000006
3,openai/gpt-5.6-terra,OpenAI: GPT-5.6 Terra,1050000,0.00000100000000000000015,0.000006
4,openai/gpt-5.6-sol-pro,OpenAI: GPT-5.6 Sol Pro,1050000,0.000005,0.00003
5,openai/gpt-5.6-sol,OpenAI: GPT-5.6 Sol,1050000,0.000005,0.00003
6,openai/gpt-chat-latest,OpenAI: GPT Chat Latest,400000,0.000005,0.00003
7,~openai/gpt-mini-latest,OpenAI GPT Mini Latest,400000,0.00000075,0.0000045
8,~openai/gpt-latest,OpenAI GPT Latest,1050000,0.000005,0.00003
9,openai/gpt-5.5-pro,OpenAI: GPT-5.5 Pro,1050000,0.00003,0.00018


## 1. Ruta Hugging Face como proveedor de modelos

Primero usaremos Hugging Face porque permite mostrar fine-tuning eficiente con PEFT.

In [5]:
# meta-llama/Llama-3.1-8B-Instruct
# mistralai/Mistral-7B-Instruct-v0.3

MODELO_HF_PRINCIPAL = "Qwen/Qwen2.5-3B-Instruct"
MODELO_HF_FALLBACK = "HuggingFaceTB/SmolLM2-360M-Instruct"

def cargar_modelo_hf(modelo_id):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(modelo_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    modelo = AutoModelForCausalLM.from_pretrained(modelo_id, torch_dtype=dtype)
    modelo.config.pad_token_id = tokenizer.pad_token_id
    modelo.eval()
    return tokenizer, modelo

try:
    MODELO_HF = MODELO_HF_PRINCIPAL
    tokenizer, modelo = cargar_modelo_hf(MODELO_HF)
except Exception as error:
    print("No se pudo cargar el modelo principal. Se usará el fallback.")
    print("Motivo:", repr(error))
    MODELO_HF = MODELO_HF_FALLBACK
    tokenizer, modelo = cargar_modelo_hf(MODELO_HF)

print("Modelo Hugging Face cargado:", MODELO_HF)
print("Tamaño del vocabulario:", len(tokenizer))
print("Parámetros:", f"{sum(p.numel() for p in modelo.parameters()):,}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Modelo Hugging Face cargado: Qwen/Qwen2.5-3B-Instruct
Tamaño del vocabulario: 151665
Parámetros: 3,085,938,688


In [ ]:
def generar_texto_hf(modelo, tokenizer, prompt, max_new_tokens=80, temperatura=0.4, top_p=0.9):
    dispositivo = next(modelo.parameters()).device
    entradas = tokenizer(prompt, return_tensors="pt").to(dispositivo)
    inicio = time.time()
    with torch.no_grad():
        salida = modelo.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=temperatura > 0,
            temperature=max(temperatura, 1e-5),
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    segundos = time.time() - inicio
    texto = tokenizer.decode(salida[0], skip_special_tokens=True)
    tokens_generados = salida.shape[-1] - entradas["input_ids"].shape[-1]
    return texto, tokens_generados, segundos

prompt = "Explica en español en 80 palabras, en tres frases, qué es cuantización en LLMs."
texto, tokens, segundos = generar_texto_hf(modelo, tokenizer, prompt)
print(f"\n{texto}")
print(f"\nTokens nuevos: {tokens} | Tiempo: {segundos:.2f} s | Tokens/s aprox.: {tokens / max(segundos, 1e-6):.2f}")


Explica en español, en tres frases, qué es cuantización en LLMs. La cuantización en LLMs es un proceso de optimización que reduce la precisión numérica de los pesos y estados ocultos de los modelos, con el objetivo de reducir el tamaño del modelo y mejorar su rendimiento en entornos de recursos limitados. Este proceso no afecta significativamente la calidad de la generación de texto, pero mejora notablemente las

Tokens nuevos: 80 | Tiempo: 121.70 s | Tokens/s aprox.: 0.66


In [7]:
experimentos_decoding = [
    {"temperatura": 0.0, "max_new_tokens": 50},
    {"temperatura": 0.4, "max_new_tokens": 50},
    {"temperatura": 0.9, "max_new_tokens": 50},
]

for cfg in experimentos_decoding:
    print("=" * 80)
    print(f"Temperatura={cfg['temperatura']} | max_new_tokens={cfg['max_new_tokens']}")
    texto, tokens, segundos = generar_texto_hf(modelo, tokenizer, prompt, **cfg)
    print(texto[:800])
    print(f"Tokens nuevos: {tokens} | Tiempo: {segundos:.2f} s")

Temperatura=0.0 | max_new_tokens=50


/home/ncaytuir/miniconda3/envs/visualization/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ncaytuir/miniconda3/envs/visualization/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/ncaytuir/miniconda3/envs/visualization/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Explica en español, en tres frases, qué es cuantización en LLMs. La cuantización en LLMs (Modelos de Lenguaje Grande) se refiere a la representación numérica de los pesos y vectores embellecidos de los modelos, reduciendo su precisión original para optimizar
Tokens nuevos: 50 | Tiempo: 83.36 s
Temperatura=0.4 | max_new_tokens=50
Explica en español, en tres frases, qué es cuantización en LLMs. La cuantización en modelos de lenguaje lingüístico (LLM) se refiere a la reducción del número de bits utilizados para representar los pesos y vectores embellecidos, lo que reduce el tamaño del modelo
Tokens nuevos: 50 | Tiempo: 83.36 s
Temperatura=0.9 | max_new_tokens=50
Explica en español, en tres frases, qué es cuantización en LLMs. La cuantización en modelos de lenguaje preentrenados (LLMs) consiste en reducir el número de valores que toman los pesos internos del modelo a valores discretos más bajos, lo que disminuye signific
Tokens nuevos: 50 | Tiempo: 84.00 s


## 2. Memoria: estimación por parámetros y precisión

Esta medición considera los tensores de parámetros. En entrenamiento real debes sumar activaciones, gradientes, estados del optimizador, caché KV y buffers temporales.

In [9]:
def resumen_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    bytes_parametros = sum(p.numel() * p.element_size() for p in modelo.parameters())
    return {
        "parametros_totales": total,
        "parametros_entrenables": entrenables,
        "porcentaje_entrenable": 100 * entrenables / total,
        "memoria_parametros_mb": bytes_parametros / 1024**2,
    }

def imprimir_resumen(nombre, modelo):
    r = resumen_parametros(modelo)
    print(nombre)
    print(f"  Parámetros totales: {r['parametros_totales']:,}")
    print(f"  Parámetros entrenables: {r['parametros_entrenables']:,}")
    print(f"  Porcentaje entrenable: {r['porcentaje_entrenable']:.4f}%")
    print(f"  Memoria de parámetros: {r['memoria_parametros_mb']:.2f} MB")

imprimir_resumen("Modelo base", modelo)

Modelo base
  Parámetros totales: 3,085,938,688
  Parámetros entrenables: 3,085,938,688
  Porcentaje entrenable: 100.0000%
  Memoria de parámetros: 5885.96 MB


In [10]:
def tabla_memoria_teorica(num_parametros):
    formatos = [
        ("FP32", 4),
        ("FP16/BF16", 2),
        ("INT8", 1),
        ("INT4", 0.5),
    ]
    filas = []
    for nombre, bytes_por_parametro in formatos:
        filas.append({
            "formato": nombre,
            "bytes_por_parametro": bytes_por_parametro,
            "memoria_solo_pesos_mb": num_parametros * bytes_por_parametro / 1024**2,
        })
    return pd.DataFrame(filas)

tabla_memoria_teorica(sum(p.numel() for p in modelo.parameters()))

,formato,bytes_por_parametro,memoria_solo_pesos_mb
0,FP32,4.0,11771.921875
1,FP16/BF16,2.0,5885.960938
2,INT8,1.0,2942.980469
3,INT4,0.5,1471.490234


## 3. Cuantización con bitsandbytes: INT8

Esta sección requiere GPU compatible y una instalación funcional de `bitsandbytes`.

In [11]:
modelo_int8 = None

if torch.cuda.is_available():
    try:
        from transformers import BitsAndBytesConfig

        config_int8 = BitsAndBytesConfig(load_in_8bit=True)
        modelo_int8 = AutoModelForCausalLM.from_pretrained(
            MODELO_HF,
            device_map="auto",
            quantization_config=config_int8,
        )
        print("Modelo cargado en INT8 con bitsandbytes.")
        imprimir_resumen("Modelo INT8", modelo_int8)
        texto, tokens, segundos = generar_texto_hf(modelo_int8, tokenizer, prompt)
        print(texto[:800])
        print(f"Tokens nuevos: {tokens} | Tiempo: {segundos:.2f} s")
    except Exception as error:
        print("No se pudo cargar INT8 en este entorno.")
else:
    print("Cuantización INT8 omitida: no hay CUDA disponible.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Modelo cargado en INT8 con bitsandbytes.
Modelo INT8
  Parámetros totales: 3,085,938,688
  Parámetros entrenables: 311,314,432
  Porcentaje entrenable: 10.0882%
  Memoria de parámetros: 3239.96 MB
Explica en español, en tres frases, qué es cuantización en LLMs. La cuantización en LLMs (Lenguajes de Lenguaje Multilingües) se refiere a la representación numérica de las palabras y conceptos para optimizar el tamaño del modelo sin perder significancia relevante. Se realiza mediante la asignación de números o códigos a cada palabra o término, reduciendo así la cantidad de información que necesita almacenarse
Tokens nuevos: 80 | Tiempo: 38.53 s


## 4. PEFT con LoRA: adaptación pequeña

Entrenaremos un adaptador LoRA con ejemplos mínimos. Esto no produce un modelo experto, pero permite observar la diferencia entre parámetros totales y parámetros entrenables.

In [12]:
from peft import LoraConfig, TaskType, get_peft_model

# Liberamos el modelo INT8 si fue cargado y no lo necesitamos ahora.
if modelo_int8 is not None:
    del modelo_int8
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

_, modelo_lora_base = cargar_modelo_hf(MODELO_HF)
modelo_lora_base.config.pad_token_id = tokenizer.pad_token_id

print("Modelo base para LoRA:", MODELO_HF)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Modelo base para LoRA: Qwen/Qwen2.5-3B-Instruct


In [14]:
def detectar_modulos_lora(modelo):
    nombres = {nombre.split(".")[-1] for nombre, _ in modelo.named_modules()}
    candidatos = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "c_attn", "c_proj",
    ]
    elegidos = [nombre for nombre in candidatos if nombre in nombres]
    if not elegidos:
        raise ValueError("No se encontraron módulos típicos para LoRA en este modelo.")
    return elegidos

target_modules = detectar_modulos_lora(modelo_lora_base)
print("Módulos elegidos para LoRA:", target_modules)

Módulos elegidos para LoRA: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


In [16]:
config_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=target_modules,
)

modelo_lora = get_peft_model(modelo_lora_base, config_lora)
modelo_lora.print_trainable_parameters()
imprimir_resumen("Modelo con adaptadores LoRA", modelo_lora)

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
Modelo con adaptadores LoRA
  Parámetros totales: 3,100,905,472
  Parámetros entrenables: 14,966,784
  Porcentaje entrenable: 0.4827%
  Memoria de parámetros: 5943.05 MB


In [17]:
ejemplos = [
    "### Instrucción: Define cuantización en LLMs.\n### Respuesta: Cuantización es reducir la precisión numérica de los pesos para ahorrar memoria y facilitar inferencia.",
    "### Instrucción: ¿Qué es PEFT?\n### Respuesta: PEFT adapta un modelo entrenando pocos parámetros adicionales y manteniendo congelado el modelo base.",
    "### Instrucción: ¿Qué hace LoRA?\n### Respuesta: LoRA agrega matrices de bajo rango entrenables en ciertas capas del modelo.",
    "### Instrucción: ¿Qué combina QLoRA?\n### Respuesta: QLoRA combina un modelo base cuantizado en 4 bits con adaptadores LoRA entrenables.",
    "### Instrucción: ¿Por qué desplegar un LLM como API?\n### Respuesta: Porque permite reutilizar el modelo desde aplicaciones, agentes y sistemas RAG sin cargarlo en cada proceso.",
    "### Instrucción: ¿Cuándo preferir RAG?\n### Respuesta: RAG conviene cuando la respuesta depende de documentos externos que cambian o no están en el modelo.",
    "### Instrucción: ¿Qué revisar antes de descargar un modelo?\n### Respuesta: Licencia, tamaño, formato, idioma, tarea, requisitos de memoria y tarjeta del modelo.",
    "### Instrucción: ¿Qué limita la inferencia local?\n### Respuesta: RAM, VRAM, velocidad de memoria, tamaño del contexto, cuantización y número de tokens generados.",
]

def preparar_batch(textos, tokenizer, max_length=160):
    batch = tokenizer(
        textos,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt",
    )
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

batch = preparar_batch(ejemplos, tokenizer)
print("Forma de input_ids:", tuple(batch["input_ids"].shape))

Forma de input_ids: (8, 46)


In [18]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo_lora.to(dispositivo)
modelo_lora.train()

batch = {k: v.to(dispositivo) for k, v in batch.items()}
optimizador = torch.optim.AdamW(modelo_lora.parameters(), lr=8e-4)

pasos = 35 if torch.cuda.is_available() else 12
historial_perdida = []
inicio = time.time()

for paso in range(1, pasos + 1):
    optimizador.zero_grad()
    salida = modelo_lora(**batch)
    perdida = salida.loss
    perdida.backward()
    optimizador.step()
    historial_perdida.append(perdida.item())
    if paso == 1 or paso == pasos or paso % 5 == 0:
        print(f"Paso {paso:02d}/{pasos} - pérdida: {perdida.item():.4f}")

print(f"Entrenamiento finalizado en {time.time() - inicio:.1f} segundos.")

Paso 01/35 - pérdida: 3.5993
Paso 05/35 - pérdida: 0.5324
Paso 10/35 - pérdida: 0.1453
Paso 15/35 - pérdida: 0.0726
Paso 20/35 - pérdida: 0.0555
Paso 25/35 - pérdida: 0.0540
Paso 30/35 - pérdida: 0.0540
Paso 35/35 - pérdida: 0.0539
Entrenamiento finalizado en 10.7 segundos.


In [19]:
modelo_lora.eval()
consulta = "### Instrucción: ¿Qué hace LoRA?\n### Respuesta:"
texto, tokens, segundos = generar_texto_hf(modelo_lora, tokenizer, consulta, max_new_tokens=70, temperatura=0.2)
print(texto)
print(f"\nTokens nuevos: {tokens} | Tiempo: {segundos:.2f} s")

### Instrucción: ¿Qué hace LoRA?
### Respuesta: LoRA agrega matrices de bajo rango entrenables en ciertas capas del modelo. Específicamente, LoRA agrega matrices de bajo rango entrenables en ciertas capas del modelo que desplegan un modelo base. Estas matrices de bajo rango se inicializan con zeros o unos y entrenables permiten

Tokens nuevos: 70 | Tiempo: 3.97 s


# Ruta Ollama: inferencia local, API, salida estructurada y mini-RAG

Ollama no reemplaza a PEFT para entrenamiento, pero sí es muy útil para:

- Probar modelos cuantizados localmente.
- Servir un LLM por HTTP desde tu equipo.
- Usar modelos desde LangChain, agentes o aplicaciones propias.
- Trabajar con embeddings locales para RAG básico.

## 5. Instalar y preparar Ollama fuera del notebook

En tu terminal local:

```bash
# 1. Instala Ollama desde la página oficial.
#    macOS, Windows y Linux: https://ollama.com/download

# Desde terminal linux:
#   curl -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst -o ollama-linux-amd64.tar.zst
#   mkdir -p "$HOME/.local/ollama"
#   tar --use-compress-program=unzstd -xf ollama-linux-amd64.tar.zst -C "$HOME/.local/ollama"
#   export PATH="$HOME/.local/ollama/bin:$PATH"
#   ollama serve # inicia el servidor

# 2. Descarga un modelo pequeño para comenzar.
ollama pull llama3.2:1b

# Alternativas livianas:
# ollama pull qwen2.5:0.5b
# ollama pull smollm2:135m

# Modelo de embeddings para mini-RAG:
ollama pull all-minilm

# 3. Prueba desde terminal.
ollama run llama3.2:1b
```

Ollama está escuchando en `http://localhost:11434`.

In [2]:
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MODELO_OLLAMA = os.environ.get("MODELO_OLLAMA", "llama3.2:1b")
MODELO_EMBEDDINGS_OLLAMA = os.environ.get("MODELO_EMBEDDINGS_OLLAMA", "all-minilm")

print("Servidor Ollama esperado:", OLLAMA_HOST)
print("Modelo de chat esperado:", MODELO_OLLAMA)
print("Modelo de embeddings esperado:", MODELO_EMBEDDINGS_OLLAMA)

Servidor Ollama esperado: http://localhost:11434
Modelo de chat esperado: llama3.2:1b
Modelo de embeddings esperado: all-minilm


In [3]:
def ollama_get(ruta, timeout=5):
    return requests.get(f"{OLLAMA_HOST}{ruta}", timeout=timeout)

def ollama_post(ruta, payload, timeout=120):
    return requests.post(f"{OLLAMA_HOST}{ruta}", json=payload, timeout=timeout)

def ollama_disponible():
    try:
        respuesta = ollama_get("/api/version", timeout=3)
        return respuesta.ok, respuesta.json()
    except Exception as error:
        return False, {"error": repr(error)}

disponible, info = ollama_disponible()
print("Ollama disponible:", disponible)
print(info)

Ollama disponible: True
{'version': '0.3.6'}


In [4]:
def listar_modelos_ollama():
    disponible, _ = ollama_disponible()
    if not disponible:
        print("Ollama no está disponible. Inicia el servicio local antes de ejecutar esta sección.")
        return []
    respuesta = ollama_get("/api/tags", timeout=10)
    respuesta.raise_for_status()
    modelos = respuesta.json().get("models", [])
    if not modelos:
        print("No hay modelos descargados. Ejecuta, por ejemplo: ollama pull llama3.2:1b")
    else:
        filas = []
        for m in modelos:
            detalle = m.get("details", {})
            filas.append({
                "nombre": m.get("name"),
                "familia": detalle.get("family"),
                "parametros": detalle.get("parameter_size"),
                "cuantizacion": detalle.get("quantization_level"),
                "tamano_gb": round(m.get("size", 0) / 1024**3, 2),
            })
        display(pd.DataFrame(filas))
    return modelos

modelos_locales = listar_modelos_ollama()

,nombre,familia,parametros,cuantizacion,tamano_gb
0,qwen2.5:0.5b,qwen2,494.03M,Q4_K_M,0.37
1,llama3.2:1b,llama,1.2B,Q8_0,1.23
2,all-minilm:latest,bert,23M,F16,0.04


## 6. Chat con Ollama vía API HTTP

Ollama expone endpoints locales. Aquí usaremos `/api/chat` con `stream=False` para recibir una respuesta completa en un solo JSON.

`stream=True`: para que el modelo genere token por token.

In [7]:
def chat_ollama(modelo, mensajes, temperatura=0.2, max_tokens=500):
    disponible, info = ollama_disponible()
    if not disponible:
        return {
            "ok": False,
            "error": "Ollama no está disponible. Revisa que el servicio esté activo en OLLAMA_HOST.",
            "detalle": info,
        }
    payload = {
        "model": modelo,
        "messages": mensajes,
        "stream": False,
        "options": {
            "temperature": temperatura,
            "num_predict": max_tokens,
        },
    }
    inicio = time.time()
    respuesta = ollama_post("/api/chat", payload, timeout=180)
    segundos = time.time() - inicio
    if not respuesta.ok:
        return {"ok": False, "status_code": respuesta.status_code, "texto": respuesta.text[:1000]}
    data = respuesta.json()
    data["segundos_cliente"] = segundos
    data["ok"] = True
    return data

mensajes = [
    {"role": "system", "content": "Responde en español, de forma precisa y breve."},
    {"role": "user", "content": "Explica la diferencia entre LoRA y QLoRA en 4 viñetas."},
]

resultado = chat_ollama(MODELO_OLLAMA, mensajes)
if resultado.get("ok"):
    #print(resultado)
    print(resultado["message"]["content"])
else:
    print(resultado)

Aquí te explico las diferencias entre LoRA (Long Range Audio) y QLoRA:

*   **Long Range Audio (LoRA)**: Es un sistema de comunicación que permite transmitir audio a largas distancias, utilizando señales de radio o satélite para llegar a una audiencia más amplia. Los sistemas de LoRA suelen utilizar tecnologías como la banda de frecuencia de 2,4 GHz y el protocolo IEEE 802.11.
*   **QLoRA**: Es un sistema de comunicación que utiliza señales de radio o satélite para transmitir audio a una audiencia más cercana, pero con menos distancia que LoRA. Los sistemas QLoRA suelen utilizar tecnologías como la banda de frecuencia de 433 MHz y el protocolo IEEE 802.11b.

En resumen, LoRA es un sistema de comunicación que permite transmitir audio a largas distancias, mientras que QLoRA es un sistema de comunicación que permite transmitir audio a una audiencia más cercana pero con menos distancia.


## 7. Parámetros de generación en Ollama

Esta sección permite ver cómo cambian las respuestas con temperatura y límite de tokens.

In [8]:
prompt_ollama = "Resume buenas prácticas para desplegar LLMs pequeños en una empresa, en español."

for temperatura in [0.0, 0.3, 0.9]:
    print("=" * 80)
    print("Temperatura:", temperatura)
    resultado = chat_ollama(
        MODELO_OLLAMA,
        [{"role": "user", "content": prompt_ollama}],
        temperatura=temperatura,
        max_tokens=120,
    )
    if resultado.get("ok"):
        print(resultado["message"]["content"])
    else:
        print(resultado)

Temperatura: 0.0
Desplegar LLM (Large Language Model) pequeños en una empresa puede ser un proceso complejo, pero con las buenas prácticas, se puede hacer de manera efectiva y segura. Aquí te presento algunas recomendaciones para desplegar LLM pequeños en tu empresa:

**1. Definir el objetivo y la misión**

Antes de desplegar cualquier modelo de lenguaje, es importante definir qué objetivo tiene y qué expectativas tienes para él. ¿Es para mejorar la comunicación con los clientes? ¿Para automatizar tareas
Temperatura: 0.3
Desplegar LLM (Large Language Model) pequeños en una empresa puede ser un proceso complejo, pero con las buenas prácticas, se puede hacer de manera efectiva y segura. Aquí te presento algunas recomendaciones para desplegar LLM pequeños en tu empresa:

**1. Definir el objetivo claro**

Antes de desplegar cualquier modelo de lenguaje, es importante definir qué propósito tiene y qué expectativas tienes. ¿Se busca mejorar la comunicación con los clientes? ¿Se busca automat

## 8. Salida estructurada con Ollama

Los sistemas RAG, agentes y pipelines de datos suelen necesitar JSON válido. Ollama permite solicitar salida estructurada mediante un esquema en el parámetro `format`, aunque la calidad depende del modelo.

In [11]:
def chat_ollama_json(modelo, prompt_usuario):
    esquema = {
        "type": "object",
        "properties": {
            "concepto": {"type": "string"},
            "definicion": {"type": "string"},
            "riesgos": {"type": "array", "items": {"type": "string"}},
            "recomendacion": {"type": "string"},
        },
        "required": ["concepto", "definicion", "riesgos", "recomendacion"],
    }
    payload = {
        "model": modelo,
        "messages": [{"role": "user", "content": prompt_usuario}],
        "stream": False,
        "format": esquema,
        "options": {"temperature": 0, "num_predict": 220},
    }
    respuesta = ollama_post("/api/chat", payload, timeout=180)
    if not respuesta.ok:
        return {"ok": False, "status_code": respuesta.status_code, "texto": respuesta.text[:1000]}
    contenido = respuesta.json()["message"]["content"]
    try:
        return {"ok": True, "json": json.loads(contenido), "contenido": contenido}
    except json.JSONDecodeError:
        return {"ok": False, "contenido": contenido, "error": "La respuesta no fue JSON válido."}

resultado_json = chat_ollama_json(MODELO_OLLAMA, "Describe QLoRA para un equipo de ciencia de datos.")
print(json.dumps(resultado_json, ensure_ascii=False, indent=2)[:2000])

{
  "ok": false,
  "status_code": 400,
  "texto": "{\"error\":\"json: cannot unmarshal object into Go struct field ChatRequest.format of type string\"}"
}


## 9. Tool calling conceptual con Ollama

Algunos modelos disponibles en Ollama soportan llamadas a herramientas. En agentes, el patrón general es:

1. El modelo decide llamar una función.
2. Tu aplicación ejecuta la función real.
3. El resultado vuelve al historial como mensaje de herramienta.
4. El modelo redacta la respuesta final.

In [12]:
def calcular_memoria_pesos(parametros_millones, bits):
    parametros_millones = float(parametros_millones)
    bits = float(bits)

    gb = parametros_millones * 1_000_000 * bits / 8 / 1024**3
    return {"memoria_gb": round(gb, 3)}

def prueba_tool_calling_ollama(modelo):
    disponible, info = ollama_disponible()
    if not disponible:
        print("Ollama no está disponible:", info)
        return

    herramienta = {
        "type": "function",
        "function": {
            "name": "calcular_memoria_pesos",
            "description": "Calcula la memoria aproximada de pesos de un modelo dado su número de parámetros y bits por parámetro.",
            "parameters": {
                "type": "object",
                "properties": {
                    "parametros_millones": {"type": "number", "description": "Parámetros en millones"},
                    "bits": {"type": "number", "description": "Bits por parámetro"},
                },
                "required": ["parametros_millones", "bits"],
            },
        },
    }

    payload = {
        "model": modelo,
        "messages": [{"role": "user", "content": "¿Cuánta memoria aproximada requieren los pesos de un modelo de 7000 millones de parámetros en 4 bits?"}],
        "tools": [herramienta],
        "stream": False,
        "options": {"temperature": 0},
    }
    respuesta = ollama_post("/api/chat", payload, timeout=180)
    if not respuesta.ok:
        print("Error HTTP:", respuesta.status_code, respuesta.text[:1000])
        return
    data = respuesta.json()
    mensaje = data.get("message", {})
    tool_calls = mensaje.get("tool_calls", [])

    if not tool_calls:
        print("El modelo no solicitó herramienta. Respuesta directa:")
        print(mensaje.get("content", ""))
        return

    print("Tool calls detectadas:")
    print(json.dumps(tool_calls, ensure_ascii=False, indent=2))

    llamada = tool_calls[0]["function"]
    args = llamada.get("arguments", {})
    resultado_funcion = calcular_memoria_pesos(**args)
    print("Resultado de la función local:", resultado_funcion)

prueba_tool_calling_ollama(MODELO_OLLAMA)

Tool calls detectadas:
[
  {
    "function": {
      "name": "calcular_memoria_pesos",
      "arguments": {
        "bits": "4",
        "parametros_millones": "7000"
      }
    }
  }
]
Resultado de la función local: {'memoria_gb': 3.26}


## 10. Mini-RAG con embeddings de Ollama

En un sistema real se usa un cargador de documentos, chunking, una base vectorial y evaluación de recuperación.

In [13]:
documentos = [
    {
        "id": "cuantizacion",
        "texto": "La cuantización reduce la precisión numérica de los pesos del modelo. Puede ahorrar memoria y facilitar inferencia, pero debe evaluarse la calidad.",
    },
    {
        "id": "peft",
        "texto": "PEFT adapta modelos grandes entrenando una pequeña cantidad de parámetros adicionales, evitando modificar todos los pesos del modelo base.",
    },
    {
        "id": "lora",
        "texto": "LoRA agrega matrices de bajo rango en capas del modelo. Estas matrices son entrenables y el modelo base puede permanecer congelado.",
    },
    {
        "id": "qlora",
        "texto": "QLoRA combina cuantización de 4 bits del modelo base con entrenamiento de adaptadores LoRA, reduciendo la memoria necesaria para adaptación.",
    },
    {
        "id": "despliegue",
        "texto": "Desplegar un LLM como servicio HTTP permite que aplicaciones, agentes y sistemas RAG lo usen sin cargar el modelo en cada proceso.",
    },
]

def embeddings_ollama(modelo_embeddings, textos):
    disponible, info = ollama_disponible()
    if not disponible:
        raise RuntimeError(f"Ollama no disponible: {info}")
    payload = {"model": modelo_embeddings, "input": textos}
    respuesta = ollama_post("/api/embed", payload, timeout=180) # endpoint
    respuesta.raise_for_status()
    return respuesta.json()["embeddings"]

def similitud_coseno(a, b):
    a = torch.tensor(a, dtype=torch.float32)
    b = torch.tensor(b, dtype=torch.float32)
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

try:
    textos_docs = [d["texto"] for d in documentos]
    emb_docs = embeddings_ollama(MODELO_EMBEDDINGS_OLLAMA, textos_docs)
    print(f"Embeddings calculados: {len(emb_docs)} documentos")
    print("Dimensión del primer embedding:", len(emb_docs[0]))
except Exception as error:
    emb_docs = None
    print("No se pudieron calcular embeddings con Ollama.")
    print("Motivo:", repr(error))

Embeddings calculados: 5 documentos
Dimensión del primer embedding: 384


In [14]:
def recuperar_documentos_ollama(pregunta, k=2):
    emb_pregunta = embeddings_ollama(MODELO_EMBEDDINGS_OLLAMA, [pregunta])[0]
    puntajes = []
    for doc, emb_doc in zip(documentos, emb_docs):
        puntajes.append((similitud_coseno(emb_pregunta, emb_doc), doc))
    puntajes = sorted(puntajes, key=lambda x: x[0], reverse=True)
    return puntajes[:k]

pregunta_rag = "¿Por qué QLoRA ayuda cuando hay poca memoria de GPU?"
recuperados = recuperar_documentos_ollama(pregunta_rag, k=2)
for puntaje, doc in recuperados:
    print(f"{doc['id']} | similitud={puntaje:.3f}\n{doc['texto']}\n")

cuantizacion | similitud=0.468
La cuantización reduce la precisión numérica de los pesos del modelo. Puede ahorrar memoria y facilitar inferencia, pero debe evaluarse la calidad.

qlora | similitud=0.441
QLoRA combina cuantización de 4 bits del modelo base con entrenamiento de adaptadores LoRA, reduciendo la memoria necesaria para adaptación.



In [16]:
def responder_con_rag_ollama(pregunta, recuperados):
    contexto = "\n".join([f"- {doc['texto']}" for _, doc in recuperados])
    mensajes = [
        {"role": "system", "content": "Responde en español usando solo el contexto entregado. Si falta información, dilo."},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"},
    ]
    resultado = chat_ollama(MODELO_OLLAMA, mensajes, temperatura=0.1, max_tokens=160)
    if resultado.get("ok"):
        print(resultado["message"]["content"])
    else:
        print(resultado)

responder_con_rag_ollama(pregunta_rag, recuperados)

La respuesta es que QLoRA permite una mayor eficiencia en el uso de la memoria del GPU, lo que reduce la carga de trabajo y facilita la inferencia. Esto se debe a que QLoRA utiliza un modelo base con 4 bits de cuantización, mientras que los adaptadores LoRA utilizan un modelo más grande y requieren una mayor cantidad de memoria para entrenar.

En resumen, QLoRA ayuda cuando hay poca memoria de GPU porque permite una mayor eficiencia en el uso de la memoria del GPU, lo que reduce la carga de trabajo y facilita la inferencia.


## Ejercicios

1. En Hugging Face, cambia `MODELO_HF_PRINCIPAL` por otro modelo.
2. Cambia `r` en LoRA de `8` a `4` y luego a `16`. ¿Cómo cambia el porcentaje entrenable?
3. Agrega cinco ejemplos nuevos al dataset de instrucciones y observa la pérdida.
4. En Ollama, prueba `qwen2.5:0.5b` y `llama3.2:1b` con el mismo prompt. Compara latencia y calidad.
5. Ejecuta el mini-RAG con una pregunta que no esté cubierta por los documentos. ¿El modelo reconoce la falta de contexto?

## Referencias y documentación

- PDFs base del diplomado: clases 5.1, 5.2 y 5.3.
- Hugging Face Model Hub: https://huggingface.co/docs/hub/models-the-hub
- Hugging Face Transformers + bitsandbytes: https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes
- Hugging Face PEFT/LoRA: https://huggingface.co/docs/peft/main/en/package_reference/lora
- Ollama Library: https://ollama.com/library
- Ollama API: https://github.com/ollama/ollama/blob/main/docs/api.md
- LM Studio Docs: https://lmstudio.ai/docs
- llama.cpp: https://github.com/ggml-org/llama.cpp
- vLLM: https://docs.vllm.ai/
- OpenRouter Quickstart: https://openrouter.ai/docs/quickstart
- OpenRouter API reference: https://openrouter.ai/docs/api/reference/overview
- OpenRouter Models API: https://openrouter.ai/docs/api/api-reference/models/get-models
